# Chapter 12 - Custom Models and Training with TensorFlow

## 1. A Quick Tour of TensorFlow

**TensorFlow** adalah library komputasi numerik yang mirip dengan **NumPy**, tetapi dilengkapi dengan fitur lanjutan seperti:
- dukungan **GPU** dan **TPU**,
- **distributed execution**,
- **automatic differentiation (autodiff)**,
- serta **Just-In-Time (JIT) compiler** berbasis *computation graph* untuk mengoptimalkan eksekusi.

Di atas *core engine* ini, TensorFlow menyediakan berbagai modul tingkat tinggi, antara lain:
- **tf.keras** untuk *deep learning* tingkat tinggi,
- **tf.data** untuk pipeline input data yang efisien,
- **tf.image** dan **tf.signal** untuk pemrosesan data khusus,
- **tf.summary** untuk visualisasi dengan **TensorBoard**,
- **TFX** untuk *production ML pipelines*,
- **TensorFlow Hub** untuk model pra-latih,
- serta **TensorFlow Lite** dan **TensorFlow.js** untuk deployment di perangkat mobile dan web.

---

### TensorFlow Architecture

Arsitektur TensorFlow memisahkan:
- **kode Python** (API tingkat tinggi seperti Keras dan Data API), dan
- **execution engine berbasis C++** yang bertugas mengeksekusi kernel pada CPU, GPU, atau TPU.

Pemisahan ini memungkinkan model untuk:
- dikembangkan dan dilatih di satu lingkungan,
- lalu dieksekusi secara efisien di lingkungan lain tanpa perubahan besar pada kode.


## 2. Custom Losses, Metrics, Initializers, Regularizers, Constraints

Bab ini membahas cara menulis **komponen kustom** pada Keras, termasuk:
- *loss functions*,
- *metrics*,
- *activation functions*,
- *initializers*,
- *regularizers*,
- serta *constraints*.

Fungsi-fungsi ini umumnya ditulis menggunakan **operasi TensorFlow**, sehingga dapat dikonversi menjadi *computation graph* dan memperoleh manfaat optimisasi serta kompatibilitas penuh dengan ekosistem TensorFlow.

---

### Custom Losses and Metrics

Salah satu contoh penting adalah implementasi **Huber loss kustom**, yang dapat ditulis:
- sebagai **fungsi sederhana**, atau
- sebagai **subclass dari `keras.losses.Loss`**, sehingga *hyperparameter* (misalnya nilai threshold) ikut tersimpan saat model di-*save*.

Pendekatan serupa dapat diterapkan pada *metrics* dengan membuat subclass dari **`keras.metrics.Metric`**, yang memungkinkan:
- akumulasi *state* antar-batch,
- penyimpanan informasi seperti jumlah *true positives* dan *false positives* secara *streaming*.

Pendekatan ini sangat berguna untuk metrik evaluasi yang kompleks dan tidak dapat dihitung dalam satu batch saja.

## **Example: Custom Huber Loss (Function & Class)**

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Functional form (simple)
def huber_fn(y_true, y_pred, threshold=1.0):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < threshold
    squared_loss = tf.square(error) / 2.0
    linear_loss  = threshold * tf.abs(error) - threshold**2 / 2.0
    return tf.where(is_small_error, squared_loss, linear_loss)

model.compile(loss=lambda y_true, y_pred: huber_fn(y_true, y_pred, 1.0),
              optimizer="nadam")

# Class form (threshold tersimpan di config)
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2.0
        linear_loss  = (self.threshold * tf.abs(error)
                        - self.threshold**2 / 2.0)
        return tf.where(is_small_error, squared_loss, linear_loss)

    def get_config(self):
        base = super().get_config()
        return {**base, "threshold": self.threshold}

model.compile(loss=HuberLoss(2.0), optimizer="nadam")


## **Example: Custom Activation / Initializer / Regularizer / Constraint**

In [ ]:
def my_softplus(z):                 # ≈ softplus
    return tf.math.log(tf.exp(z) + 1.0)

def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2.0 / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)

def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01 * weights))

def my_positive_weights(weights):   # enforce non-negative
    return tf.where(weights < 0., tf.zeros_like(weights), weights)

layer = keras.layers.Dense(
    30,
    activation=my_softplus,
    kernel_initializer=my_glorot_initializer,
    kernel_regularizer=my_l1_regularizer,
    kernel_constraint=my_positive_weights,
)


## 3. Custom Layers

Jika dibutuhkan perilaku layer yang tidak tersedia secara langsung di Keras, terdapat beberapa pendekatan yang dapat digunakan:

- **Lambda layer**  
  Digunakan untuk operasi sederhana tanpa bobot, misalnya fungsi `exp` atau transformasi element-wise lainnya.

- **Subclass `keras.layers.Layer`**  
  Digunakan untuk membuat layer dengan bobot atau perilaku yang lebih kompleks, mirip dengan implementasi layer `Dense` kustom.

---

### Implementing a Custom Layer

Subclass dari `keras.layers.Layer` umumnya mengimplementasikan metode berikut:
- `__init__()` untuk menyimpan *hyperparameter*
- `build()` untuk membuat bobot layer menggunakan `add_weight()`
- `call()` untuk mendefinisikan proses *forward pass*
- `get_config()` agar layer dapat di-*save* dan di-*load* dengan benar

Untuk layer yang berperilaku berbeda antara **training** dan **inference** (seperti **Dropout**, **Batch Normalization**, atau **Noise layers**), metode `call()` menerima argumen `training` dan menyesuaikan perilakunya berdasarkan mode tersebut.


## **Example: Simple Custom Dense Layer**

In [ ]:
from tensorflow import keras
import tensorflow as tf

class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=[batch_input_shape[-1], self.units],
            initializer="glorot_normal"
        )
        self.bias = self.add_weight(
            name="bias",
            shape=[self.units],
            initializer="zeros"
        )
        super().build(batch_input_shape)

    def call(self, X):
        z = X @ self.kernel + self.bias
        return self.activation(z) if self.activation is not None else z

    def get_config(self):
        base = super().get_config()
        return {
            **base,
            "units": self.units,
            "activation": keras.activations.serialize(self.activation),
        }

# Use it like a normal layer
model = keras.models.Sequential([
    keras.layers.InputLayer(input_shape=(10,)),
    MyDense(30, activation="relu"),
    MyDense(1)
])


## **Example: Custom Gaussian Noise Layer (Different at Training/Testing)**

In [ ]:
class MyGaussianNoise(keras.layers.Layer):
    def __init__(self, stddev, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, X, training=None):
        if training:
            noise = tf.random.normal(tf.shape(X), stddev=self.stddev)
            return X + noise
        return X

    def get_config(self):
        base = super().get_config()
        return {**base, "stddev": self.stddev}


## 4. Custom Models & Losses Based on Internals

Untuk arsitektur yang lebih kompleks—misalnya yang melibatkan **residual blocks**, **auxiliary heads**, atau alur komputasi yang tidak linear—model dapat dibuat dengan melakukan **subclassing terhadap `keras.Model`**.

Pada pendekatan ini:
- `__init__()` digunakan untuk mendefinisikan layer atau blok penyusun model
- `call()` mengatur alur *forward pass*, termasuk perulangan, *skip connections*, dan *multi-output*

---

### Internal-State-Based Losses

Keras juga memungkinkan penambahan **loss tambahan** yang bergantung pada *internal state* model, seperti aktivasi atau bobot tertentu. Loss semacam ini dapat dimasukkan ke total loss menggunakan metode `add_loss()`.

Pendekatan ini berguna untuk:
- regularisasi berbasis aktivasi,
- constraint struktural,
- atau tujuan optimisasi tambahan yang tidak langsung berasal dari label target.

## **Example: Residual Block & ResidualRegressor**

In [ ]:
class ResidualBlock(keras.layers.Layer):
    def __init__(self, n_layers, n_neurons, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [
            keras.layers.Dense(
                n_neurons, activation="elu", kernel_initializer="he_normal"
            )
            for _ in range(n_layers)
        ]

    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        return inputs + Z  # skip connection


class ResidualRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(
            30, activation="elu", kernel_initializer="he_normal"
        )
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out = keras.layers.Dense(output_dim)

    def call(self, inputs):
        Z = self.hidden1(inputs)
        for _ in range(1 + 3):
            Z = self.block1(Z)
        Z = self.block2(Z)
        return self.out(Z)


## **Example: Custom Reconstruction Loss via add_loss**

In [ ]:
class ReconstructingRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [
            keras.layers.Dense(
                30, activation="selu", kernel_initializer="lecun_normal"
            )
            for _ in range(5)
        ]
        self.out = keras.layers.Dense(output_dim)

    def build(self, batch_input_shape):
        n_inputs = batch_input_shape[-1]
        self.reconstruct = keras.layers.Dense(n_inputs)
        super().build(batch_input_shape)

    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        reconstruction = self.reconstruct(Z)
        recon_loss = tf.reduce_mean(tf.square(reconstruction - inputs))
        self.add_loss(0.05 * recon_loss)  # auxiliary regularization loss
        return self.out(Z)


## 5. Autodiff with tf.GradientTape & Custom Gradients

TensorFlow menyediakan **reverse-mode automatic differentiation** melalui **`tf.GradientTape`**, yang memungkinkan perhitungan gradien fungsi kompleks terhadap banyak variabel secara efisien.

Di dalam blok `with tf.GradientTape()`, TensorFlow akan merekam semua operasi yang melibatkan `tf.Variable`. Setelah itu, metode `tape.gradient()` digunakan untuk menghitung gradien dari suatu nilai skalar (misalnya *loss*) terhadap satu atau lebih variabel.

---

### Advanced Usage of GradientTape

Beberapa fitur penting dari `GradientTape` meliputi:
- **Persistent tape**  
  Dengan `persistent=True`, gradien dapat dihitung lebih dari sekali dari tape yang sama.
- **Watching tensors**  
  Tape dapat secara eksplisit mem-*watch* tensor biasa (bukan `tf.Variable`) jika gradien terhadap input diperlukan, misalnya untuk regularisasi berbasis sensitivitas.
- **Custom gradients**  
  Untuk kasus numerik yang sulit atau tidak stabil, dekorator **`@tf.custom_gradient`** memungkinkan pendefinisian rumus gradien secara manual agar proses pelatihan lebih stabil.

Pendekatan ini memberikan fleksibilitas tinggi untuk eksperimen optimisasi tingkat lanjut.


## **Example: Using GradientTape**

In [ ]:
import tensorflow as tf

def f(w1, w2):
    return 3 * w1**2 + 2 * w1 * w2

w1 = tf.Variable(5.0)
w2 = tf.Variable(3.0)

with tf.GradientTape() as tape:
    z = f(w1, w2)

grads = tape.gradient(z, [w1, w2])
print(grads[0].numpy(), grads[1].numpy())  # 36.0, 10.0


## **Example: Custom Gradient with @tf.custom_gradient**

In [ ]:
@tf.custom_gradient
def my_better_softplus(z):
    exp = tf.exp(z)
    y = tf.math.log(exp + 1.0)

    def grad(dy):
        return dy / (1.0 + 1.0 / exp)  # stable derivative

    return y, grad


## 6. Custom Training Loops & TensorFlow Functions

Meskipun metode **`model.fit()`** sudah mencakup sebagian besar kebutuhan pelatihan (*sekitar 95% use case*), terdapat situasi di mana **custom training loop** diperlukan, misalnya:
- menggunakan dua optimizer berbeda untuk bagian jaringan yang berbeda,
- menerapkan aturan pembaruan gradien khusus,
- atau melakukan logging dan monitoring yang sangat kustom.

---

### Writing a Custom Training Loop

Dengan **`tf.GradientTape`**, loop pelatihan manual biasanya mencakup langkah-langkah berikut:
1. Mengambil satu batch data
2. Melakukan *forward pass*
3. Menghitung *total loss* (termasuk regularisasi dari `model.losses`)
4. Menghitung gradien
5. Menerapkan gradien dengan optimizer
6. Memperbarui metrik secara manual

Pendekatan ini memberikan kontrol penuh atas setiap langkah pelatihan.

---

### TensorFlow Functions (`tf.function`)

Dekorator **`tf.function`** mengubah fungsi Python menjadi **TensorFlow Function** yang dieksekusi sebagai *optimized computation graph* melalui **AutoGraph** dan *tracing*.

Beberapa aturan penting dalam penggunaan `tf.function`:
- Gunakan **operasi TensorFlow**, bukan NumPy murni
- Hindari *side-effect* Python yang memengaruhi logika program
- Buat variabel **di luar** TF Function
- Gunakan **`tf.range`** untuk perulangan yang ingin ditangkap ke dalam graph

Dengan mengikuti aturan ini, TensorFlow dapat menghasilkan eksekusi yang lebih cepat dan efisien.

## **Example: Custom Training Loop (Regression with Nadam)**

In [ ]:
import numpy as np
from tensorflow import keras
import tensorflow as tf

# Model
l2_reg = keras.regularizers.l2(0.05)
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu",
                       kernel_initializer="he_normal",
                       kernel_regularizer=l2_reg),
    keras.layers.Dense(1, kernel_regularizer=l2_reg)
])

# Random mini-batch sampler
def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

# Status bar
def print_status_bar(iteration, total, loss, metrics=None):
    metrics = " - ".join(
        "{}: {:.4f}".format(m.name, m.result())
        for m in [loss] + (metrics or [])
    )
    end = "" if iteration < total else "\n"
    print("\r{}/{} - {}".format(iteration, total, metrics), end=end)

n_epochs = 5
batch_size = 32
n_steps = len(X_train) // batch_size
optimizer = keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = keras.losses.mean_squared_error

mean_loss = keras.metrics.Mean(name="loss")
metrics = [keras.metrics.MeanAbsoluteError(name="mae")]

for epoch in range(1, n_epochs + 1):
    print("Epoch {}/{}".format(epoch, n_epochs))
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train_scaled, y_train, batch_size)
        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)  # include reg losses
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        mean_loss.update_state(loss)
        for metric in metrics:
            metric.update_state(y_batch, y_pred)

        print_status_bar(step * batch_size, len(X_train), mean_loss, metrics)

    print_status_bar(len(X_train), len(X_train), mean_loss, metrics)
    for m in [mean_loss] + metrics:
        m.reset_states()


## **Example: Converting to TF Function**

In [ ]:
@tf.function
def train_step(X_batch, y_batch):
    with tf.GradientTape() as tape:
        y_pred = model(X_batch, training=True)
        main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
        loss = tf.add_n([main_loss] + model.losses)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    mean_loss.update_state(loss)
    for metric in metrics:
        metric.update_state(y_batch, y_pred)
